# Introducción a la Ciencia de Datos: Tarea 1

Este notebook contiene el código de base para realizar la Tarea 1 del curso. Puede copiarlo en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y la librería Pandas. Si no tiene ninguna familiaridad con la librería, se recomienda realizar algún tutorial introductorio (ver debajo).
También se espera que los alumnos sean proactivos a la hora de consultar las documentaciones de las librerías y del lenguaje, para entender el código provisto.
Además de los recursos provistos en la [página del curso](https://eva.fing.edu.uy/course/view.php?id=1378&section=1), los siguientes recursos le pueden resultar interesantes:
 - [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) y [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): Son parte de la documentación en la página oficial de Pandas.
 - [Kaggle Learn](https://www.kaggle.com/learn): Incluye tutoriales de Python y Pandas.


Si desea utilizar el lenguaje R y está dispuesto a no utilizar (o traducir) este código de base, también puede hacerlo.

En cualquier caso, **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook (ver [README](https://github.com/DonBraulio/introCD)).

In [ ]:
from time import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datasets import load_dataset

# Agregue aqui el resto de las librerias que necesite
# from ...
# import ...

## Descarga del dataset
En esta tarea se utilizará una base de datos abierta que contiene artículos de noticias publicados en distintos medios de prensa, con la finalidad de realizar una clasificación de textos según el medio de prensa al que pertenecen. [Link](https://huggingface.co/datasets/rjac/all-the-news-2-1-Component-one?utm_source=chatgpt.com) \
\
Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas. La constante `DATA_PATH` determina la ubicación donde se almacenaran los datos. \
\
El dataset entero pesa ~8.3gb. Para evitar demoras en la descarga/procesamiento vamos a utilizar el parámetro `streaming=True` y hacer un muestreo aleatorio para descargar una porción de los datos lo más representativa posible.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train",cache_dir="../data")
df = ds.to_pandas()

## Lectura de Datos

In [ ]:
# Veamos las primeras filas del DataFrame
df.head()

In [ ]:
# Veamos información general del DataFrame
df.info()

In [ ]:
df_reuters = df[df['publication'] == 'Reuters']
df_reuters_without_word_Reuters = df_reuters[~df_reuters['article'].str.contains('Reuters', case=False, na=False)]
display(df_reuters)

# Parte 1: Cargado y Limpieza de Datos

## A. Exploración de Datos
Analice el contenido del DataFrame. Reporte si existen datos faltantes en algún campo, o cualquier otro problema de calidad de datos que encuentre. \
En particular, analice la cantidad de artículos por medio de prensa, y a partir de este punto trabaje con los **cinco medios con mayor cantidad de artículos**.

In [ ]:
# TODO: Analice datos faltantes por columna
# Returns a count of missing values for every column
df.isna().sum()



In [ ]:
# TODO: Analice la cantidad de artículos por medio de prensa

# Tome los 5 medios con más artículos
top5 = df['publication'].dropna().value_counts().head(5).index
print(top5)
# top_5_publications = ...
df_top_5 = df[df['publication'].isin(top5)]
len(df_top_5)

## B. Visualización temporal
Genere una gráfica que permita visualizar los artículos de los cinco medios a lo largo del tiempo, con alguna escala temporal adecuada. \
Comente si se identifican momentos de mayor actividad o patrones temporales en la cobertura.

In [ ]:
# TODO: Visualización de los artículos de cada medio a lo largo del tiempo
# Preste especial atención al formato de la columna 'date', ya que puede contener diferentes formatos de fecha.

df['date'] = pd.to_datetime(df['date'], errors='coerce', format= 'mixed')
df['date'].isna().sum()

# Recreate df_top_5 after date conversion to ensure dates are datetime objects
df_top_5 = df[df['publication'].isin(top5)]
df_top_5 = df_top_5[df_top_5['date'].notna()]
df_top_5['month'] = df_top_5['date'].dt.to_period('M')

counts = (
    df_top_5
    .groupby(['month', 'publication'])
    .size()
    .unstack(fill_value=0)
)

counts.index = counts.index.to_timestamp()

counts = counts[:-1]
counts.plot(figsize=(10,6))

Momento de mayor actividad: A partir de enero de 2020. Razones politicas.

## C. Limpieza de texto y conteo de palabras
Se provee la función `clean_text(...)` que realiza parte de la normalización del texto. \
**Complete la función** agregando signos de puntuación faltantes y cualquier otra normalización que considere oportuna. \
Compruebe el resultado observando el contenido del DataFrame procesado. Comente todas las transformaciones que haya agregado y justifique.

In [ ]:
def clean_text(df, column_name):

    # Eliminar primeras palabras hasta el primer "\n"
    result = df[column_name].str.replace(r"^[^\n]*\n", "", regex=True)

    # Convertir todo a minúsculas
    result = result.str.lower()

    # TODO: completar signos de puntuación faltantes
    for punc in ["[", "\n", ",", ":", "?", "!", "(", ")", '"', "'"]:
        result = result.str.replace(punc, " ")

    return result

In [ ]:
# TODO: Aplique clean_text sobre la columna de texto elegida y cree una nueva columna "CleanText"

CleanText = clean_text(df_top_5, "article")
print(CleanText.head(n=25))

you minglei is a name but it can be confused with the pronoun "you"
is it ok to keep the reuters header?
some of them start with a space
maybe apply the clean_text to title?
"." is ok not to remove cause U.S could be confused with pronoun "us"

## D. Elección de campos de texto
Discuta si conviene trabajar con:
- sólo el cuerpo del artículo,
- sólo el título,
- o una combinación de ambos.

Justifique brevemente su decisión.

*TODO: Escriba su análisis en el informe.*

Conviene trabjar con una combinacion de ambos pues la eleccion de palabras es muy importante en el titulo para captar la atencion del lector. 

## E. Pistas que identifican al medio de prensa
Analice si en el texto aparecen pistas que identifiquen de manera directa al medio de prensa (nombres del medio, URLs, firmas, nombres de secciones, plantillas repetidas, etc.). \
En caso de encontrarlas, comente si considera conveniente eliminarlas o reducir su impacto, y justifique su decisión.

In [ ]:
# TODO: Explore el texto buscando pistas que identifiquen directamente al medio de prensa
# Por ejemplo, busque nombres de medios, URLs, firmas, etc.

In [ ]:
# Reuters 

df_reuters = df[df['publication'] == 'Reuters']
display(len(df_reuters))

df_reuters_with_word_reuters = df_reuters[df_reuters['article'].str.contains('Reuters', case=False, na=False)]
display(len(df_reuters_with_word_reuters))
display(df_reuters_with_word_reuters)

df_reuters_with_word_brief = df_reuters[df_reuters['title'].str.contains('brief', case=False, na=False)]
display(len(df_reuters_with_word_brief))
display(df_reuters_with_word_brief)

In [ ]:
# People

df_people = df[df['publication'] == 'People']
display(df_people)

Parece ser que People solo escribe articulos sobre personas famosas. Los titulos siempre empiezan con el nombre y apellido de la persona en cuestion, o quizas un "How", "When" y luego el nombre de la persona. 

Se podria armar una lista de paises, y chequear que las tres primeras palabras empiecen con mayuscula y a su vez no se refieran a paises. 

O encontrar una lista de las personas mas mencionadas en los medios y usarla para descubir si alguna de las tres primeras palabras del articulo coinciden con un item de dicha lista. 

La hora aparece como parte de la date. Esto cuenta?

In [ ]:
# CNBC
# Configure pandas to show full text content
pd.set_option('display.max_colwidth', 100)

df_cnbc = df[df['publication'] == 'CNBC']
# display(df_cnbc)

display(df_cnbc['article'])
display(len(df_cnbc))
df_cnbc_with_no_author = df_cnbc[df_cnbc['author'].isna()]
display(len(df_cnbc_with_no_author))

Observation: The word Reuters appear in some articles like referencing the other company:
(Updates with new first paragraph, adds background) March 29 (Reuters) 

Casi ningun articulo tiene el autor especificado

## F. Restricción por sección o período temporal
Evalúe si conviene restringir el análisis a artículos de una misma sección temática o de un período temporal acotado, con el objetivo de reducir el efecto del tema sobre una futura tarea de clasificación por medio. \
No es necesario implementar esta restricción, pero sí discutir sus posibles ventajas y desventajas.

*TODO: Escriba su análisis en el informe.*

Si, segun los periodos marcados en la grafica. 
La seccion tematica habla mas de famosos, quiza no serviria la comparativa. Habria que analizar las tematicas tocadas por cada medio.

# Parte 2: Conteo de Palabras y Visualizaciones

## A. Palabras más frecuentes por medio
Realice una visualización que permita comparar las palabras más frecuentes de cada uno de los cinco medios de prensa. \
Sin necesidad de implementarlo, proponga ideas para modificar esta visualización con el fin de encontrar diferencias entre secciones temáticas, fechas, o tipos de noticias.

In [ ]:
# TODO: Realice una visualización que permita comparar las palabras más frecuentes
# de cada uno de los cinco medios de prensa.
# - ¿Encuentra algún problema en los resultados?

import nltk 

nltk.download('stopwords')
def get_most_frequent_words(text_series, n=20):
    # Concatenate all text into a single string
    all_text = ' '.join(text_series)

    # Tokenize the text into words
    words = nltk.word_tokenize(all_text)

    # Remove stopwords and punctuation
    stopwords = set(nltk.corpus.stopwords.words('english'))
    words = [word for word in words if word.isalpha() and word not in stopwords]

    # Get the frequency distribution of the words
    freq_dist = nltk.FreqDist(words)

    # Return the n most common words and their frequencies
    return freq_dist.most_common(n)

## B. Medios con mayor cantidad de palabras
Corra el código que permite encontrar los medios con mayor cantidad de palabras. \
En caso de encontrar algún problema luego de realizar la visualización, comente a qué se debe y proponga formas de resolverlo.

In [ ]:
# TODO: Busque los medios con mayor cantidad de palabras

## C. Matriz de menciones entre medios
Construya una matriz de 5×5, donde cada fila y columna corresponden a un medio de prensa, y la entrada (i,j) contiene la cantidad de veces que el medio *i* menciona al medio *j*. \
\
**Opcional:** genere un grafo dirigido con esa matriz de adyacencia para visualizar las menciones. Puede ser útil la biblioteca `networkx`.

In [ ]:
# TODO: Construya una matriz de 5x5, donde cada fila y columna corresponden a un medio de prensa,
# y la entrada (i,j) contiene la cantidad de veces que el medio "i" menciona al medio "j".

# mentions_matrix = ...


In [ ]:
# Opcional: Genere un grafo dirigido con la matriz de adyacencia para visualizar las menciones.
# Puede ser útil la biblioteca networkx.



## D. Preguntas propuestas
Proponga al menos tres preguntas que se podrían intentar responder a partir de estos datos, y mencione posibles caminos para responderlas, sin implementar nada.

*TODO: Escriba sus preguntas y posibles caminos en el informe.*